# Week 2, Day 2: Agent Orchestration

## What this lab covers

This lab first creates a Windows Outlook email tool, then explores orchestration approaches for multiple agents. The code below coordinates agents directly in Python before showing model-led delegation through tools and handoffs.

#### Part 01: Email Setup
#### Part 02: Orchestrating by code
#### Part 03: Orchestrating by LLMs
 - ##### 3a: via Tools
 - ##### 3b: via Handoffs

### Part 01: Email Sender Tool Setup

In [1]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_disabled
from openai import AsyncAzureOpenAI 
from agents.model_settings import ModelSettings
from IPython.display import Markdown, display
from agents.extensions.visualization import draw_graph
import asyncio
import datetime
import win32com.client
import pythoncom
import subprocess
import re
import time

set_tracing_disabled(True)  # Disable tracing for this notebook to avoid cluttering the output
load_dotenv(override=True) # Load environment variables from .env file, override existing ones if necessary

True

In [2]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
import os
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")

# From Anthropic
AZURE_CLAUDE_DEPLOYMENT_OPUS_48=os.getenv("AZURE_CLAUDE_DEPLOYMENT_OPUS_48")
AZURE_CLAUDE_ENDPOINT=os.getenv("AZURE_CLAUDE_ENDPOINT")
AZURE_CLAUDE_API_KEY=os.getenv("AZURE_CLAUDE_API_KEY")
if AZURE_CLAUDE_API_KEY:
    print("AZURE_CLAUDE_API_KEY is avaiable")
else:
    print("AZURE_CLAUDE_API_KEY is not available")

# Loading Email addresses
EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS_TO")
if EMAIL_ADDRESS: 
    print("Email address found!")
else:
    print("Email address not found!")

AZURE_OPENAI_API_KEY is available
AZURE_CLAUDE_API_KEY is avaiable
Email address found!


In [3]:
## setting up the client for OpenAI using Azure Endpoint (Azure Foundry)
client_openai = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)
## let's a define a model pointing to Azure deployment
model_openai = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_54_mini
)

In [4]:
## Setting up email sending method/tool 
# define a method/function to send email via Outlook COM
@function_tool
def send_email_tool( 
    subject: str,
    body: str
    ) -> bool:
    """Send an email using the local Outlook desktop client."""
    # Normalize recipient to a semicolon-separated string (Outlook format)
    recipient = EMAIL_ADDRESS
    recipients = []
    if isinstance(recipient, list):
        recipients = [r.strip() for r in recipient if r.strip()]
    elif isinstance(recipient, str):
        if "," in recipient:
            recipients = [r.strip() for r in recipient.split(",") if r.strip()]
        else:
            recipients = [recipient.strip()]
    
    def validate_email(email: str) -> bool:
        """Validate email format."""
        pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$'
        return re.match(pattern, email.strip()) is not None

    # Validate all recipients
    invalid_recipients = [r for r in recipients if not validate_email(r)]
    if invalid_recipients:
        print(f"Error: Invalid email format(s): {invalid_recipients}")
        return False
    
    recipient_str = "; ".join(recipients)
    
    if not recipient_str:
        print("Error: No recipients specified. Email not sent.")
        return False

    if body is None:
        body = "Hello there!! "

    print(f"\nSending email to {recipient_str} with subject '{subject}'...")

    # Ensure COM is initialized for the current thread (agent tool calls may run in worker threads).
    pythoncom.CoInitialize()
    try:
        # Connect to Outlook, launch it if not running
        outlook = None
        for attempt in range(5):
            try:
                outlook = win32com.client.Dispatch("Outlook.Application")
                outlook.GetNamespace("MAPI")
                print("Outlook is up & running...")
                break
            except Exception as e:
                if attempt == 0:
                    print("Outlook not running — launching...")
                    for p in [r"C:\Program Files\Microsoft Office\root\Office16\OUTLOOK.EXE",
                              r"C:\Program Files (x86)\Microsoft Office\root\Office16\OUTLOOK.EXE"]:
                        if os.path.exists(p):
                            subprocess.Popen([p])
                            time.sleep(45)
                            break
                    else:
                        print("Error: Outlook executable not found.")
                        return False
                elif attempt < 4:
                    print(f"Outlook connect attempt {attempt + 1}/5 — waiting...")
                    time.sleep(15)
                else:
                    print(f"Error: Outlook connect failed after 5 attempts: {e}")
                    return False

        try:
            mail = outlook.CreateItem(0)
            mail.To = recipient_str
            mail.Subject = subject
            mail.Body = body

            mail.Send()
            print(f"Email sent successfully to '{recipient_str}' with subject '{subject}'.")
            return True

        except Exception as e:
            print(f"Failed to send email: {e}")
            # Print recipient info for debugging
            print(f"Debug - Recipients used: {repr(recipient_str)}")
            print(f"Debug - Recipient type: {type(recipient_str)}")
            return False
    finally:
        try:
            pythoncom.CoUninitialize()
        except Exception:
            pass

In [5]:
### Let's try sending one simple email via above created method
# send_email_tool("Agentic Project Testing 02", "Hello there!!")

In [6]:
# let's create some custom tools with Python using OpenAI SDK 
# @function_tool # help's making json format, required for AI model to understand about the function/tool
def record_message_tool(topic: str, message: str) -> str:
    """ Record the user message with it's topic in a file  """
    print(f"Tool called to record an message: '{message}' on topic '{topic}'")
    # setting up the timestamp and entry format
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"\n\n ## [{timestamp}] - TOPIC: {topic}\n* **Content:** {message}"

    with open("ai_model_message.txt","a", encoding="utf-8") as file:
        file.write(entry)
    return f"Successfully saved message on topic '{topic}'."

### Part 02: Orchestrating by Code

In [7]:
# Let's set up a system prompt
intro = """
You are a sales agent working for a company called 'CommodiPulse',
a company which delivers lag-free, real-time data feeds for global commodities including power, carbon, agriculture, oil, and gas.
We empower businesses with instant, synchronized market insights to make critical trading and supply decisions with absolute certainty.
"""

professional_instructions = intro + "Your email style is professional, serious, with gravitas and credibility."
humorous_instructions = intro + "Your email style is witty, engaging, and humorous."
executive_instructions = intro + "Your email style is concise, to the point, in the style of a busy senior executive."



In [8]:
## now let's set up agents for sending emails
professional_agent = Agent(
    name = "Professional Sales Agent", 
    instructions = professional_instructions,
    model = model_openai
)
humorous_agent = Agent(
    name = "Humorous Sales Agent", 
    instructions = humorous_instructions,
    model = model_openai
)
executive_agent = Agent(
    name = "Executive Sales Agent", 
    instructions = executive_instructions,
    model = model_openai
)

In [11]:
## Let's run the agent and see results
result = Runner.run_streamed(
    starting_agent = professional_agent, 
    input = "Write a cold sales email"
)
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush = True)
    

Subject: Real-time commodity data, without the lag

Hi [First Name],

I’m reaching out from CommodiPulse.

We provide lag-free, real-time data feeds across power, carbon, agriculture, oil, and gas—giving trading, risk, and supply teams a synchronized view of the market as it moves.

In fast-moving commodity markets, even small delays can create uncertainty in pricing, hedging, and operational decisions. Our platform is designed to remove that friction by delivering instant, reliable market insight you can act on with confidence.

If timely, accurate commodity intelligence is important to your team, I’d welcome a brief conversation to understand your current setup and see whether CommodiPulse could add value.

Would you be open to a 15-minute call next week?

Best regards,  
[Your Name]  
[Title]  
CommodiPulse  
[Phone] | [Email] | [Website]

In [12]:
## Let's run all of agents in parallel 
message = "Write a cold sales email"
results = await asyncio.gather(
    Runner.run(professional_agent, message),
    Runner.run(humorous_agent, message),
    Runner.run(executive_agent, message)
)

In [15]:
map_agent = {
    0: "professional_agent",
    1: "humorous_agent",
    2: "executive_agent"
}
outputs = [result.final_output for result in results]

for index, output in enumerate(outputs):
    print(f"\n\nAnswer from '{map_agent.get(index)}': \n\n {output}")




Answer from 'professional_agent': 

 Subject: Real-time commodities data, without the lag

Hi [First Name],

I’m reaching out from CommodiPulse.

We provide lag-free, real-time data feeds across global commodities markets — including power, carbon, agriculture, oil, and gas — so trading, procurement, and risk teams can act on synchronized market intelligence with confidence.

In markets where minutes matter, delayed or fragmented data can distort decisions, pricing, and exposure. CommodiPulse is designed to eliminate that uncertainty by delivering a single, accurate view of the market as it moves.

If you’re responsible for trading, supply, or market analytics, I’d welcome the opportunity to show you how firms are using CommodiPulse to improve decision speed and reduce risk.

Would you be open to a brief 15-minute conversation next week?

Best regards,  
[Your Name]  
[Title]  
CommodiPulse  
[Email] | [Phone] | [Website]


Answer from 'humorous_agent': 

 Subject: Real-time commodit

In [9]:
## Let's define a new agent called 'sales_picker'
decision = """
You pick the best cold sales email from the given options. 
Imagine you are a customer and pick the one you are most likely to respond to.
Do not give an explanation; reply with the selected email only.
"""
sales_picker = Agent(
    name = "Sales_Picker",
    instructions = decision,
    model = model_openai
)

In [18]:
# let's run above set agent to select the best email
emails = f"Cold sales emails: \n\n\n Email: \n\n {outputs}"
best_email = await Runner.run(sales_picker, emails)
print(f"Best email: \n\n{best_email.final_output}")

Best email: 

Subject: Real-time commodities data, without the lag

Hi [First Name],

I’m reaching out from CommodiPulse.

We provide lag-free, real-time data feeds across power, carbon, agriculture, oil, and gas — giving trading and supply teams instant, synchronized market insight when timing matters most.

If your team relies on fast-moving commodity markets, we can help reduce delay and improve confidence in critical decisions.

Open to a brief conversation next week?

Best,  
[Your Name]  
CommodiPulse


In [31]:
# Let's give a check on email sending tool
send_email_tool.params_json_schema

{'properties': {'subject': {'title': 'Subject', 'type': 'string'},
  'body': {'title': 'Body', 'type': 'string'}},
 'required': ['subject', 'body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [10]:
 ## Let's define a new agent called 'sales_sender'
decision = """
You pick the best cold sales email from the given options. 
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email from 'Hardeep Singh' CEO of company and answer with the context and confirmation of sent email. No extra text.
"""

strict_settings = ModelSettings(
    tool_choice="required",
    temperature=0.0
)
sales_sender = Agent(
    name="Sales Sender",
    instructions=decision, 
    model=model_openai,
    tools = [send_email_tool],
    model_settings = strict_settings
    )


In [40]:
## Let's run all of agents in parallel and let the sales-sender use tool to send the most appropriate email 
message = "Write a cold sales email"
results = await asyncio.gather(
    Runner.run(professional_agent, message),
    Runner.run(humorous_agent, message),
    Runner.run(executive_agent, message)
)
# create mapper to the agents results
map_agent = {
    0: "professional_agent",
    1: "humorous_agent",
    2: "executive_agent"
}
outputs = [result.final_output for result in results]

for index, output in enumerate(outputs):
    print(f"\n\n#####---------Answer from '{map_agent.get(index)}'------------#####: \n\n {output}")

# let's run above set agent to select the best email
emails = f"Cold sales emails: \n\n\n Email: \n\n {outputs}"
best_email = await Runner.run(
    starting_agent = sales_sender,
    input = emails
)
print(f"##### -------------- Best Email ----------------##### \n\n{best_email.final_output}")



#####---------Answer from 'professional_agent'------------#####: 

 Subject: Real-time commodities data you can act on with confidence

Hi [First Name],

I’m reaching out from CommodiPulse.

We provide lag-free, real-time data feeds across power, carbon, agriculture, oil, and gas — giving trading, risk, and supply teams instant, synchronized market insight when timing matters most.

In volatile commodity markets, even small delays can lead to missed opportunities, weaker execution, and avoidable risk. Our platform is built to remove that uncertainty and help teams make faster, more confident decisions with trusted market data.

If improving the speed and reliability of your commodity intelligence is a priority, I’d welcome a brief conversation to see whether CommodiPulse could be relevant for your team.

Would you be open to a 15-minute call next week?

Best regards,  
[Your Name]  
[Title]  
CommodiPulse  
[Email] | [Phone]


#####---------Answer from 'humorous_agent'------------###

#### Part 03: Orchestrating by LLMs
##### - 3a. Via Tools

In [11]:
description = "Use this tool to write a sales email. In the input, just instruct it to write a sales email."

professional_agent_tool = professional_agent.as_tool(
    tool_name="sales_email_writer_1", 
    tool_description=description
)
professional_agent_tool


FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000027FB7D90DF0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

In [12]:
professional_agent_tool = professional_agent.as_tool(
    tool_name="sales_email_writer_1", 
    tool_description=description
)
humorous_agent_tool = humorous_agent.as_tool(
    tool_name="sales_email_writer_2", 
    tool_description=description
)
executive_agent_tool = executive_agent.as_tool(
    tool_name="sales_email_writer_3", 
    tool_description=description
)
agent_tools = [professional_agent_tool, humorous_agent_tool, executive_agent_tool, send_email_tool]
agent_tools

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000027FB7D908E0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None),
 FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties'

In [13]:
## Let's prepare a prompt for a sales manager who would have some tools (auto agents) to be used and complete his work.
instructions = """
You are a Sales Manager at a company called 'CommodiPulse'. Your goal is to find the single best cold sales email using the sales_writer tools.
"""
task = """
Follow these steps:
1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. Evaluate and Select: Review the drafts and choose the single best email using your judgement of which one is most effective.

3. Use your tool to send the single best email found in last step to the user.
"""
sales_manager = Agent(
    name = "Sales Manager",
    instructions=instructions,
    tools=agent_tools,
    model=model_openai
)

In [28]:
result = await Runner.run(
    starting_agent = sales_manager,
    input = task
)

print(f"Sales manager agent's results: \n{result.final_output}")


Sending email to hardeep.singh3@lseg.com with subject 'Real-time commodities data you can rely on'...
Outlook is up & running...
Email sent successfully to 'hardeep.singh3@lseg.com' with subject 'Real-time commodities data you can rely on'.
Sales manager agent's results: 
Done — I selected the strongest draft and sent it.


#### Part 03: Orchestrating by LLMs
##### - 3b via Handoffs

In [14]:
professional_agent_tool = professional_agent.as_tool(
    tool_name="sales_email_writer_1", 
    tool_description=description
)
humorous_agent_tool = humorous_agent.as_tool(
    tool_name="sales_email_writer_2", 
    tool_description=description
)
executive_agent_tool = executive_agent.as_tool(
    tool_name="sales_email_writer_3", 
    tool_description=description
)
agent_tools = [professional_agent_tool, humorous_agent_tool, executive_agent_tool, send_email_tool]
agent_tools

[FunctionTool(name='sales_email_writer_1', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x0000027FB7D935B0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None),
 FunctionTool(name='sales_email_writer_2', description='Use this tool to write a sales email. In the input, just instruct it to write a sales email.', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties'

In [15]:
 ## Let's define a new agent called 'sales_sender'
decision = """
You pick the best cold sales email from the given options. 
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email from 'Hardeep Singh' CEO of company to a customer named 'charles' and answer with the context and confirmation of sent email. No extra text.
"""

strict_settings = ModelSettings(
    tool_choice="required",
    temperature=0.0
)
sales_sender = Agent(
    name="Sales Sender",
    instructions=decision, 
    model=model_openai,
    tools = [send_email_tool],
    model_settings = strict_settings
    )


In [16]:
## Let's prepare a prompt for a sales manager who would have some tools (auto agents) to be used and complete his work.
instructions = """
You are a Sales Manager at a company called 'CommodiPulse'. Your get your sales team to draft emails, then send them all to a sales picker.
"""
task = """
Follow these steps:
1. Generate Drafts: Use each of the three sales_email_writer tools to generate different email drafts.
Just instruct each to write a sales email; no further details are needed.
Do not proceed until all three drafts are ready, one from each tool.

2. handoff to the sales sender to choose and send the best email to user.
"""
# setting up the handoffs agent
handoffs = [sales_sender]

sales_manager = Agent(
    name = "Sales Manager",
    instructions=instructions,
    tools=agent_tools,
    model=model_openai,
    handoffs=handoffs
)

In [17]:
result = await Runner.run(
    starting_agent = sales_manager,
    input = task
)
print(f"Sales Manager results:\n {result.final_output}")


Sending email to hardeep.singh3@lseg.com with subject 'Real-time commodities data, without the lag'...
Outlook is up & running...
Email sent successfully to 'hardeep.singh3@lseg.com' with subject 'Real-time commodities data, without the lag'.
Sales Manager results:
 Drafts reviewed. Sent to Charles: Subject “Real-time commodities data, without the lag” from Hardeep Singh, CEO, CommodiPulse. Confirmation: sent successfully.
